<a href="https://colab.research.google.com/github/Sergi-e/lab-4-llm-decision-support/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# API key set up
from google.colab import userdata
API_KEY = userdata.get('GROQ_API_KEY')

# # TODO: set API_KEY using ONE of the methods above.
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


# Section 1 — Talking to an LLM Programmatically

In [ ]:
# Part 1.1 — My first API call

# TODO: helper function I'll reuse for the whole lab
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

In [ ]:
# TODO: call it once with a simple question and print the answer
answer = ask_llm("What is a good name for a savings product for market traders in Accra?")
print(answer)

For a savings product targeting market traders in Accra, you'll want a name that resonates with their needs, is easy to remember, and reflects the local culture. Here are some suggestions:

1. **Sika Kokoo**: "Sika" means "money" in the Akan language, and "Kokoo" means "gather" or "collect." This name conveys the idea of gathering savings.
2. **Market Mmoa**: "Mmoa" means "savings" or "treasury" in Akan. This name is straightforward and targets the market trader audience.
3. **Akwaaba Savings**: "Akwaaba" is a popular Ghanaian phrase meaning "welcome." This name creates a sense of inclusivity and warmth.
4. **Traders' Treasure**: This name emphasizes the idea of saving for the future and positions the product as a valuable resource for market traders.
5. **Susu Plan**: "Susu" is a traditional Ghanaian savings system where individuals contribute to a collective fund. This name leverages the existing cultural practice and adds a modern twist.
6. **Makola Mmoa**: "Makola" is a well-known 

In [ ]:
# TODO: print response.usage to see how many tokens the call consumed
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is a good name for a savings product for market traders in Accra?"},
    ],
)
print(response.usage)

CompletionUsage(completion_tokens=404, prompt_tokens=57, total_tokens=461, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.153106628, prompt_time=0.005217627, completion_time=1.550786896, total_time=1.556004523)


> **Student Reasoning — Anatomy of a call:**
>
> **1. System vs user roles:** The system role sets the model's identity and rules for the whole conversation, while the user role is the actual request being made in that turn. In my ask_llm() function, the system role tells the model to act as a helpful assistant, and the user role asks the specific question about a savings product name for market traders in Accra.
>
> **2. Tokens and billing:** A token is roughly a piece of a word, sometimes a whole short word and sometimes just a few characters, that the model reads and generates one at a time. My test call used 57 tokens for the prompt and 398 for the completion, for a total of 455. Providers bill per token instead of per request because the actual computing cost depends on how much text is processed and generated, not on the fact that a request was made. A one word answer and an eight paragraph list, like the one I got back, cost very different amounts to produce, so token based billing reflects that difference fairly.

In [ ]:
# Part 1.2 — temperature: the randomness dial
# TODO: ask the same question 5 times at temperature=0.0 and 5 times at temperature=1.2
savings_question = "Suggest a name for a savings product for market traders in Accra."

low_temp_runs = [ask_llm(savings_question, temperature=0.0) for _ in range(5)]
high_temp_runs = [ask_llm(savings_question, temperature=1.2) for _ in range(5)]

In [ ]:
# TODO: print all 10 answers, grouped by temperature
print("Temperature 0.0 runs")
for i, ans in enumerate(low_temp_runs, 1):
    print(f"Run {i}: {ans}")

print("\nTemperature 1.2 runs")
for i, ans in enumerate(high_temp_runs, 1):
    print(f"Run {i}: {ans}")

Temperature 0.0 runs
Run 1: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Sika Souce**: "Sika" means "money" in the Akan language, which is widely spoken in Ghana. "Souce" is a play on the word "source," implying a reliable and trustworthy savings product.
4. **Market Mobi**: This name incorporates "mobi," short for mobile, to convey the idea of a convenient and accessible savings product.
5. **Adanbo Save**: "Adanbo" is a Ghanaian word that means "progress" or "prosperity," which could appeal to market traders looking to improve their financial situation.
6. **Kokroko Savings**: "Kokroko" is a Ghanaian word that means "honest" or "trustworthy," which could help build confidence in the savings product.
7. **Traders' Fund**: This name is straightforwa

> **Student Reasoning — Temperature:**
>
> **1. What I observed:** At temperature 0.0, the answers were nearly identical across runs, the same names kept repeating, and two runs came back word for word the same. At temperature 1.2, the answers varied a lot more, new names appeared that never showed up at temperature 0, and even small details like word meanings shifted between runs.
>
> **2. Which temperature fits the loan decision-support system:** Temperature 0.0 is the right choice for that system. A loan officer needs consistent, repeatable output when reviewing an application, not a different summary or recommendation each time the same letter is processed. High temperature is useful for brainstorming, like generating product name ideas, but it works against reliability in a decision-support context, where trust depends on the system giving the same read on the same facts every time.

# Section 2 — The Dataset: Loan Application Letters

In [ ]:
# TODO: load the six loan application letters and gold-standard labels
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


# Section 3 — Prompt Engineering for the Decision Support System

In [ ]:
# Part 3.1 — Component 1: Summarization
# TODO: naive first attempt, run on L002 and L006
def summarize_v1(letter_text):
    return ask_llm(f"Summarize this: {letter_text}")

summary_l002_v1 = summarize_v1(LETTERS["L002"])
summary_l006_v1 = summarize_v1(LETTERS["L006"])

print("L002 V1:", summary_l002_v1)
print("\nL006 V1:", summary_l006_v1)

L002 V1: Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and promises to repay the loan as soon as possible, despite not having collateral.

L006 V1: Kofi, a 22-year-old, is requesting GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan in one year, once his businesses are successful, but has no collateral to offer, relying on his personal trustworthiness.


In [ ]:
# TODO: proper V2 template with a role and constraints, run at temperature=0
summary_system_prompt = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Summarize loan applications factually and neutrally, in 3 to 4 sentences. "
    "Do not invent or assume details that are not stated in the letter."
)

def summarize_v2(letter_text):
    user_prompt = f"Summarize this loan application:\n\n{letter_text}"
    return ask_llm(user_prompt, system_prompt=summary_system_prompt, temperature=0)

summary_l002_v2 = summarize_v2(LETTERS["L002"])
summary_l006_v2 = summarize_v2(LETTERS["L006"])

print("L002 V2:", summary_l002_v2)
print("\nL006 V2:", summary_l006_v2)

L002 V2: Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but expects it to improve after the festive season. He does not currently have collateral to offer, but is seeking urgent assistance with the loan.

L006 V2: Kofi is applying for a loan of GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business. He is 22 years old and claims to be business-minded, citing feedback from his friends. Kofi has not yet started any of these businesses and plans to repay the loan in one year. He does not have collateral to offer, but asserts that he is trustworthy.


In [ ]:
# TODO: compare V1 vs V2 side by side
print("L002 comparison")
print("V1:", summary_l002_v1)
print("V2:", summary_l002_v2)

print("\nL006 comparison")
print("V1:", summary_l006_v1)
print("V2:", summary_l006_v2)

L002 comparison
V1: Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and promises to repay the loan as soon as possible, despite not having collateral.
V2: Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng mentions that his business has been slow, but expects it to improve after the festive season. He does not currently have collateral to offer, but is seeking urgent assistance with the loan.

L006 comparison
V1: Kofi, a 22-year-old, is requesting GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the 

> **Student Reasoning — Summarization prompts:**
>
> **1. Problems in V1 that V2 fixed:** V1 added small framing not actually in the letter. For L002, V1 wrote that Kwame "promises to repay the loan as soon as possible," but the letter says "I can pay back whenever the money comes," which is a much less certain statement. For L006, V1 wrote that Kofi promises to repay "once his businesses are successful," which is another confident phrase the letter never uses. V2 stayed closer to the literal wording in both cases.
>
> **2. Why "no invented details" is necessary, and the failure mode name:** A wording that sounds more certain or positive than the source could push a loan officer toward trusting an application more than what the letter actually supports. This is called hallucination in the LLM literature: when a model states something that is not grounded in the source text. Since the officer is making real financial decisions off this output, I think keeping it strictly factual is a good safeguard.

In [ ]:
# Part 3.2 — Component 2: structured extraction
# TODO: prompt with schema, one few-shot example not from LETTERS, null if not stated, temperature=0
extract_system_prompt = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Extract loan application details as a JSON object with exactly these keys: "
    "applicant_name (string), amount_ghs (number), purpose (string), "
    "monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean), "
    "repayment_months (number or null). "
    "If a field is not stated in the letter, use null. Do not guess. "
    "Return only the JSON object, nothing else."
)

extract_few_shot_example = """Example letter:
"My name is Ama Asante, I run a small chop bar in Cape Coast. I need GHS 5,000 to buy a new fridge. My monthly profit is around GHS 600. My husband will guarantor the loan. I will repay over 10 months."

Example JSON output:
{"applicant_name": "Ama Asante", "amount_ghs": 5000, "purpose": "buy a new fridge", "monthly_profit_ghs": 600, "has_collateral_or_guarantor": true, "repayment_months": 10}
"""

In [ ]:
# TODO: extract_fields, strip json fences, json.loads, handle parse failures gracefully
import json

def extract_fields(letter_text):
    user_prompt = extract_few_shot_example + f"\n\nNow extract from this letter:\n{letter_text}"
    raw_output = ask_llm(user_prompt, system_prompt=extract_system_prompt, temperature=0)

    cleaned = raw_output.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.replace("json", "", 1).strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print(f"Could not parse output for this letter: {raw_output}")
        return None

In [ ]:
# TODO: run on all six letters, collect into a pandas DataFrame
import pandas as pd

extracted_rows = []
for letter_id, letter_text in LETTERS.items():
    fields = extract_fields(letter_text)
    if fields is not None:
        fields["letter_id"] = letter_id
        extracted_rows.append(fields)

extracted_df = pd.DataFrame(extracted_rows)
extracted_df

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months,letter_id
0,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0,L001
1,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN,L002
2,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0,L003
3,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0,L004
4,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0,L005
5,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0,L006


> **Student Reasoning — Structured extraction:**
>
> **1. Why the few-shot example can't come from the six letters:** If the example came from one of the six letters, the model could pattern-match against data it had already seen instead of applying the extraction format to new data. Using a made-up example, like Ama Asante's chop bar loan, tests whether the model understood the schema rather than memorized an answer.
>
> **2. What "use null, do not guess" prevents:** Letters like L002 and L006 don't mention a monthly profit figure. Without this instruction, the model might fill the gap with a plausible but made-up number, which is risky in a system used for financial decisions. The instruction makes the model leave those fields as null, matching what the letters actually say.
>
> **3. Why temperature 0 fits extraction but not creative tasks:** Extraction needs the same correct answer for the same input because the officer is pulling exact facts and numbers, not generating ideas. Temperature 0 keeps the output consistent. A creative task like naming a savings product benefits from variation, so a higher temperature works better, but that variation would be a problem for extraction.

In [ ]:
# Part 3.3 — Component 3: decision-support brief
# TODO: BRIEF_PROMPT takes the letter and extracted JSON, outputs strengths, risks,
#   missing info, and a suggested next step. No approve/reject. Humans decide.
brief_system_prompt = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Given a loan application letter and its extracted data, write a decision-support brief with: "
    "1. Strengths (bullet points, grounded in the letter). "
    "2. Risks or red flags (bullet points). "
    "3. Missing information the officer should request. "
    "4. A suggested next step, such as 'invite for interview', 'request documents', or 'flag for senior review'. "
    "Never output 'approve' or 'reject'. Final decisions are made by a human loan officer, not by you."
)

def generate_brief(letter_text, extracted_json):
    user_prompt = (
        f"Loan application letter:\n{letter_text}\n\n"
        f"Extracted data:\n{json.dumps(extracted_json)}\n\n"
        "Write the decision-support brief."
    )
    return ask_llm(user_prompt, system_prompt=brief_system_prompt, temperature=0)

In [ ]:
# TODO: generate briefs for all six letters, print L001, L002, L006
briefs_by_letter = {}
for letter_id, letter_text in LETTERS.items():
    fields = extracted_df[extracted_df["letter_id"] == letter_id].iloc[0].to_dict()
    briefs_by_letter[letter_id] = generate_brief(letter_text, fields)

for letter_id in ["L001", "L002", "L006"]:
    print(f"{letter_id} brief:\n{briefs_by_letter[letter_id]}\n")
# TODO: also print L003 for the strong vs weak comparison
print(f"L003 brief:\n{briefs_by_letter['L003']}")

L001 brief:
**Decision-Support Brief**

**Strengths:**
* The applicant, Akosua Mensah, has a long-standing business experience of 12 years selling provisions at Makola Market.
* She has a stable monthly profit of GHS 900, which indicates a consistent income stream.
* Akosua has demonstrated a good savings habit through the susu scheme, with GHS 2,500 saved over two years without missing a contribution.
* She has a guarantor, her sister, who is a teacher, providing an added layer of security for the loan.
* The applicant has provided a clear plan for loan repayment, with a proposed monthly repayment amount of GHS 450 over 20 months.

**Risks or Red Flags:**
* The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit, which may pose a risk if the business expansion does not generate sufficient additional income.
* There is no information provided about the applicant's current debt obligations or credit history, which could impact her ability to repay the loan

**Student Reasoning — Decision Support:**

**1. Comparing L003 and L006:** The system found clear strengths in L003: a registered business, verifiable sales records, fixed-deposit collateral, and a proven revenue record. Its main risk was cash-flow pressure from the loan size, not whether the business was legitimate. L006 had weaker strengths, mainly enthusiasm and a diverse business idea, with little concrete evidence to support it. Its risks were more serious: none of the businesses had started, there was no collateral, and repayment depended on the assumption that things would go well. The system identified the key difference: a proven business facing financial pressure versus an unproven idea with no safety net.

**2. Why we forbid "approve"/"reject":** A wrong automated loan decision can have direct financial consequences for both the applicant and the institution. The system also cannot verify information outside the letter, such as credit history or business registration. A human loan officer can request documents and check what the model cannot. Ethically, allowing AI to make final lending decisions removes accountability from a process that can affect someone's livelihood. These applicants are running businesses and supporting families, so the final decision should come from a person who can be held responsible, not just a model output.

### Part 3.4 — Commit your prompt templates

> **Commit hash:** 286a04e

# Section 4 — Evaluation

In [ ]:
# Part 4.1 — accuracy check against gold-standard labels
# TODO: compare extracted rows for L001, L003, L006 against GOLD, field by field
gold_letter_ids = list(GOLD.keys())
field_names = ["applicant_name", "amount_ghs", "purpose", "monthly_profit_ghs",
               "has_collateral_or_guarantor", "repayment_months"]

match_records = []
for letter_id in gold_letter_ids:
    extracted_row = extracted_df[extracted_df["letter_id"] == letter_id].iloc[0]
    gold_row = GOLD[letter_id]
    for field in field_names:
        extracted_value = extracted_row[field]
        gold_value = gold_row[field]

        # names can differ in case, so compare loosely, everything else must match exactly
        if field == "applicant_name" and isinstance(extracted_value, str) and isinstance(gold_value, str):
            is_match = extracted_value.strip().lower() == gold_value.strip().lower()
        else:
            is_match = extracted_value == gold_value

        match_records.append({"field": field, "letter_id": letter_id, "match": is_match})

match_df = pd.DataFrame(match_records)

In [ ]:
# TODO: small table, rows = fields, columns = L001 / L003 / L006 / accuracy
# pivot turns the long match_df into one row per field, one column per letter
summary_table = match_df.pivot(index="field", columns="letter_id", values="match")
summary_table["accuracy"] = summary_table[gold_letter_ids].mean(axis=1)
summary_table

letter_id,L001,L003,L006,accuracy
field,,,,
amount_ghs,True,True,True,1.000000
applicant_name,True,True,True,1.000000
has_collateral_or_guarantor,True,True,True,1.000000
monthly_profit_ghs,True,True,False,0.666667
purpose,False,False,False,0.000000
repayment_months,True,True,True,1.000000


In [ ]:
# Part 4.2 — reliability check on L004
# TODO: run extract_fields five times at temperature=0 and five times at temperature=1.0
def extract_fields(letter_text, temperature=0):
    user_prompt = extract_few_shot_example + f"\n\nNow extract from this letter:\n{letter_text}"
    raw_output = ask_llm(user_prompt, system_prompt=extract_system_prompt, temperature=temperature)

    cleaned = raw_output.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.replace("json", "", 1).strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print(f"Could not parse output for this letter: {raw_output}")
        return None

low_temp_extractions = [extract_fields(LETTERS["L004"], temperature=0) for _ in range(5)]
high_temp_extractions = [extract_fields(LETTERS["L004"], temperature=1.0) for _ in range(5)]

In [ ]:
# TODO: for each temperature, count valid JSON runs and identical-value runs
def count_valid_and_identical(runs):
    valid_runs = [r for r in runs if r is not None]
    # sort_keys makes dict comparison order-independent, so equal dicts produce equal strings
    unique_strings = set(json.dumps(r, sort_keys=True) for r in valid_runs)
    return len(valid_runs), len(unique_strings)

low_valid, low_unique = count_valid_and_identical(low_temp_extractions)
high_valid, high_unique = count_valid_and_identical(high_temp_extractions)

print(f"Temperature 0: {low_valid}/5 valid JSON, {low_unique} unique result(s)")
print(f"Temperature 1.0: {high_valid}/5 valid JSON, {high_unique} unique result(s)")

Temperature 0: 5/5 valid JSON, 1 unique result(s)
Temperature 1.0: 5/5 valid JSON, 1 unique result(s)


In [ ]:
# Part 4.3 — Test 1: ask about a detail that is not in the letter
# TODO: ask a question about missing info, see if the model admits it's absent or invents one
credit_score_question = ask_llm(
    f"What is the applicant's credit score?\n\nLetter:\n{LETTERS['L001']}",
    system_prompt=summary_system_prompt,
    temperature=0
)
print(credit_score_question)

The letter does not mention the applicant's credit score. Akosua Mensah is applying for a loan of GHS 8,000 to expand her business and has a history of saving with the susu scheme, making regular contributions over the past two years. She has proposed a repayment plan of GHS 450 per month for 20 months. Her sister, a teacher, has agreed to stand as her guarantor.


In [ ]:
# Part 4.3 — Test 2: feed the extractor something irrelevant
# TODO: run extract_fields on an unrelated text, see if it returns nulls or fabricates an applicant
weather_report = (
    "Weather update for Accra: partly cloudy skies today with a high of 31 degrees Celsius. "
    "Expect scattered showers in the afternoon, clearing up by evening. Humidity remains high "
    "at around 80 percent."
)
weather_extraction = extract_fields(weather_report, temperature=0)
print(weather_extraction)

{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


**Adversarial test results**

Test 1 (missing detail - credit score): The model responded "The letter does not mention the applicant's credit score" instead of inventing one. **PASS**

Test 2 (irrelevant input - weather report): extract_fields() returned all null values (applicant_name: None, amount_ghs: None, purpose: None, monthly_profit_ghs: None, has_collateral_or_guarantor: None, repayment_months: None) instead of fabricating an applicant. **PASS**

> **Student Reasoning — Evaluation results:**
>
> **1. Extraction accuracy:** Five of six fields scored 100% across L001, L003, and L006. purpose scored 0%, but that's because it's free text, my wording matched the letter's meaning but not gold's exact phrasing, so exact-match comparison counts it as wrong even when the content is right. monthly_profit_ghs also showed one false match for L006, caused by None (gold) and NaN (pandas) not comparing equal, even though both mean "not stated."
>
> **2. Reliability:** Both temperature 0 and temperature 1.0 gave 5/5 valid JSON and only 1 unique result across all 5 runs. I expected temperature 1.0 to add variation, like it did with the open-ended question in Part 1.2. It didn't here, likely because the strict schema and instructions constrain the model enough that even higher temperature doesn't change the output for a well-defined task like extraction.
>
> **3. Hallucination probing:** The system passed both tests. Asked about a credit score not in the letter, it said plainly the letter doesn't mention one instead of inventing a number. Fed a weather report, the extractor returned all null fields instead of fabricating an applicant. If it had failed either test, tightening the "do not guess" instruction or adding a check for irrelevant input before extraction would reduce the risk further.

**Student Reasoning — Appropriateness:**

**1. Who could be unfairly harmed by full automation:** Applicants with solid businesses but weak English or rushed, informal writing, like L002 or L006, could be penalized even when their businesses are doing well. Since the model judges confidence and detail from the writing, someone with a real, profitable business but a short, vague letter could score worse than someone with weaker finances but a polished application. This could affect less-educated applicants most, even though they are among those microfinance is meant to serve.

**2. Sending personal data to a third-party API abroad:** These letters contain names, income, and financial details that would be sent to a US-based provider after leaving Ghana. Before using the system, I would check the provider's data retention policy, whether Ghana's Data Protection Act requires local storage or consent for this transfer, and whether submitted data is used for training. Applicants should also know that their information will be processed by a foreign AI system before submitting their letters.

**3. Two concrete safeguards:** First, require a human review before any decision or next step reaches the applicant, so the AI only supports the decision and does not act alone. Second, keep an audit log of each brief and the officer's final decision. This would help identify and correct bias, such as L002/L006-style applicants being repeatedly flagged for extra scrutiny.

**Section 5 — Reflection:**

**1. Prompting as engineering:** Iterating on a prompt is similar to tuning hyperparameters in Lab 3 because both involve trial and error: change something, observe the output, and adjust. However, prompting changes the input without changing the model, and results come in seconds instead of after a training run. Prompt changes can also be less predictable. A hyperparameter change may produce a measurable accuracy shift, while a prompt can improve one letter and hurt another, so broader testing is needed.

**2. Trust:** I would not trust this system to run fully unattended. The hallucination tests in Part 4.3 influenced my answer most. Although the system passed both, two successful tests do not prove it will never fail. Part 4.4 also showed that applicants who write differently but run solid businesses could still be harmed. Therefore, keeping a human in the loop, as the system does by refusing to output approve/reject, is the right approach.

**3. Cost and scale:** My Part 1.1 test used 57 prompt tokens and 398 completion tokens, 455 total, for one simple question. Since summarize, extract, and brief each make a separate call per letter, I estimate about 300–800 tokens per call, or roughly 1,500 tokens per letter across all three. For 1,000 applications a month, that is about 1.5 million tokens. At that volume, Groq's free tier would likely need to be upgraded, so provider choice becomes more important at real-world scale.

**4. API versus training your own model:** Using an API makes more sense here because the task requires general language understanding, such as reading varied loan letters and reasoning about their content. Foundation models already handle this well after training on far more data and computing resources than I could access for this project. Training my own model would require a large labeled dataset and significant computing resources. However, a custom model could make more sense for a narrower task, such as classifying letters into a few fixed categories, where a smaller model could be cheaper to run at scale.